# Сравнение DL-моделей для тональности отзывов

**Цель:** выбрать модель для подключения к ML-сервису reo. Обе модели — готовые русскоязычные классификаторы на три класса; оцениваем их на одном тестовом сплите и одинаково обрабатываем предсказания.

**Решение по этому сравнению:** подключать `r1char9/rubert-base-cased-russian-sentiment`. На общем тестовом сплите он показал macro-F1 **0.8142** против **0.7970** у обязательной для сравнения `seara/rubert-tiny2-russian-sentiment` (+0.0172), и F1 выше во всех классах. `seara` остается более компактной альтернативой, если по результатам нагрузочного замера базовая модель не уложится в ограничения сервиса.


## Данные и протокол

- Датасет: [MTEB RuReviewsClassification](https://huggingface.co/datasets/mteb/RuReviewsClassification), split `test`.
- В тестовом split **2 048** русскоязычных товарных отзывов; классы сбалансированы: negative — 682, neutral — 683, positive — 683.
- Истинные метки: `0 = negative`, `1 = neutral`, `2 = positive`. Для моделей метки сопоставляются по текстовым именам, поэтому внутренний порядок классов в конфигурации модели не влияет на оценку.
- Для обеих моделей одинаковы тексты, `max_length=512`, batch size 32 и метрики precision, recall, F1 и support по классам. Основная метрика выбора — macro-F1.

**Ограничение:** карточки обеих моделей указывают RuReviews среди обучающих корпусов. Поэтому возможное пересечение обучающих данных с источником тестового сплита нельзя исключить. Результат полезен для первичного выбора между моделями, но не является гарантией качества на независимых данных reo. Перед финальной эксплуатацией стоит дополнительно проверить отложенную выборку отзывов reo.


In [ ]:
import gc

from datasets import load_dataset
from sklearn.metrics import classification_report, f1_score
from transformers import pipeline

DATASET_ID = "mteb/RuReviewsClassification"
MODEL_IDS = [
    "seara/rubert-tiny2-russian-sentiment",
    "r1char9/rubert-base-cased-russian-sentiment",
]
CLASS_NAMES = ["negative", "neutral", "positive"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
BATCH_SIZE = 32
MAX_LENGTH = 512
DEVICE = 0 if __import__("torch").cuda.is_available() else -1


In [2]:
test_data = load_dataset(DATASET_ID, split="test")
texts = list(test_data["text"])
y_true = list(test_data["label"])

assert len(texts) == len(y_true) == 2048
print(f"Dataset: {DATASET_ID} / test; n={len(texts)}")
print("Классы:", {name: y_true.count(i) for i, name in enumerate(CLASS_NAMES)})


Dataset: mteb/RuReviewsClassification / test; n=2048
Классы: {'negative': 682, 'neutral': 683, 'positive': 683}


In [3]:
reports = {}
predictions_by_model = {}

for model_id in MODEL_IDS:
    print(f"Оцениваем {model_id} ...")
    classifier = pipeline(
        "text-classification",
        model=model_id,
        device=DEVICE,
    )
    predictions = classifier(
        texts,
        batch_size=BATCH_SIZE,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    # Переводим имена классов модели к общему порядку: negative, neutral, positive.
    y_pred = [LABEL_TO_ID[item["label"].lower()] for item in predictions]
    predictions_by_model[model_id] = y_pred
    reports[model_id] = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    print(f"macro-F1 = {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
    del classifier
    gc.collect()


Оцениваем seara/rubert-tiny2-russian-sentiment ...
macro-F1 = 0.7970
              precision    recall  f1-score   support

    negative      0.8045    0.7302    0.7656       682
     neutral      0.6806    0.7643    0.7200       683
    positive      0.9199    0.8917    0.9056       683

    accuracy                          0.7954      2048
   macro avg      0.8017    0.7954    0.7970      2048
weighted avg      0.8017    0.7954    0.7971      2048

Оцениваем r1char9/rubert-base-cased-russian-sentiment ...
macro-F1 = 0.8142
              precision    recall  f1-score   support

    negative      0.8230    0.7361    0.7771       682
     neutral      0.7013    0.7906    0.7433       683
    positive      0.9326    0.9122    0.9223       683

    accuracy                          0.8130      2048
   macro avg      0.8190    0.8130    0.8142      2048
weighted avg      0.8190    0.8130    0.8142      2048



In [4]:
print(f"{'Модель':58} {'Macro-F1':>9} {'Neg F1':>9} {'Neu F1':>9} {'Pos F1':>9}")
for model_id in MODEL_IDS:
    report = reports[model_id]
    print(
        f"{model_id:58} {report['macro avg']['f1-score']:9.4f} "
        f"{report['negative']['f1-score']:9.4f} "
        f"{report['neutral']['f1-score']:9.4f} "
        f"{report['positive']['f1-score']:9.4f}"
    )


Модель                                                     Macro-F1    Neg F1    Neu F1    Pos F1
seara/rubert-tiny2-russian-sentiment                         0.7970    0.7656    0.7200    0.9056
r1char9/rubert-base-cased-russian-sentiment                  0.8142    0.7771    0.7433    0.9223


## Выбор для сервиса

Подключаем **`r1char9/rubert-base-cased-russian-sentiment`**: на одной и той же размеченной выборке macro-F1 выше на **1.72 п.п.** (0.8142 против 0.7970), при этом F1 улучшился для negative (0.7771 против 0.7656), neutral (0.7433 против 0.7200) и positive (0.9223 против 0.9056).

Модель `seara/rubert-tiny2-russian-sentiment` сохраняем как компактный fallback. Базовая модель крупнее, поэтому перед подключением измерить на целевом CPU потребление памяти и задержку для типичной пачки; выбранная модель остается основной по качеству на этом сравнении. Метрики и вывод относятся только к указанному split и требуют подтверждения на независимых данных reo.

Источники: [модель seara](https://huggingface.co/seara/rubert-tiny2-russian-sentiment), [модель r1char9](https://huggingface.co/r1char9/rubert-base-cased-russian-sentiment), [датасет](https://huggingface.co/datasets/mteb/RuReviewsClassification).
